# Test 04 — Incorrect train/test split
Top comment: Uses a much shorter OOS window and uses future data when constructing benchmark - incorrect train/test split.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
p = Path.cwd() / 'data' / 'GW05_original_monthly.csv'
df = pd.read_csv(p, sep=';', decimal=',')
df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')
df.set_index('date', inplace=True)
df['equity_premium'] = df['CRSP_SPvw'] - df['Rfree']
# Wrong: use est_periods_OOS=12 (1-year) and include future rows in training by mistake
def compute_bad_split(ts, var):
    x = ts[var].shift(1).dropna()
    y = ts.loc[x.index, 'equity_premium']
    reg = OLS(y, add_constant(x)).fit()
    return {'IS_R2_head': reg.rsquared_adj*100, 'OOS_R2_head': 5.0}
vars_list = ['dp','dy','ep']
results = {v: compute_bad_split(df, v) for v in vars_list}
import pandas as pd
df_results = pd.DataFrame.from_dict(results, orient='index')
df_results.index.name = 'variable'
df_results